# 02 — Stabl Feature Selection

**Pipeline Step 2 of 5**

This notebook applies **Stabl** (Stability Selection with L1-penalised Logistic Regression) to objectively identify the most reproducible biomarker genes that distinguish **TBI** from **Sham** conditions.

### Why Stabl?

Traditional gene selection methods (fold-change thresholds, p-value cutoffs) produce unstable gene lists. Stabl addresses this by bootstrapping the dataset 500 times, fitting an L1-penalised model on each subsample, and retaining only genes consistently selected across iterations. The FDP+ framework sets an automatic, data-driven threshold with controlled false-discovery guarantees.

### Two-stage feature selection: Significance → Stability

Instead of the naive "top 2,000 HVGs" approach, we use a **t-test differential expression pre-filter** with BH FDR correction to first select genes that are *statistically significantly different* between TBI and Sham. This focused gene set then goes into Stabl for *stability* assessment.

**Why t-test over logistic regression for the pre-filter?**
- t-test provides **real p-values** with Benjamini-Hochberg FDR correction
- Logistic regression coefficients lack native p-values — would need permutation testing
- Using LR as a pre-filter is **redundant** since Stabl already uses L1-LogReg internally
- t-test is **complementary**: statistical significance (DE) → reproducibility (Stabl)

### Pipeline (condition-mode)
1. **Downsample** ≤ 1,000 spots per sample (balanced, reproducible)
2. **ComBat batch correction** across 6 samples
3. **t-test DE pre-filter** (FDR < 0.01, |log₂FC| > 0.5)
4. **Condition labels** assigned from ground-truth metadata (TBI=1, Sham=0)
5. **500-bootstrap Stabl** with FDP+ thresholding

### Inputs
| File | Description |
|---|---|
| `data/processed/tbi_preprocessed.h5ad` | QC-filtered, normalized AnnData from Step 01 |

### Outputs
| File | Description |
|---|---|
| `cache/stabl_results_<hash>.pkl` | Full Stabl result dictionary |
| `cache/stabl_features_<hash>.csv` | Table of selected genes with stability scores |

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.spatial_pipeline import load_adata, run_stabl_cached

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
CACHE_DIR = PROJECT_ROOT / "cache"

print("Imports ready.")

Imports ready.


## 2.1 Load Preprocessed Data

Load the QC-filtered and normalized AnnData produced by notebook 01 (6 samples, ~24.9k spots).

In [2]:
adata = load_adata(DATA_PROCESSED / "tbi_preprocessed.h5ad")
print(f"\nLoaded: {adata.shape[0]} spots × {adata.shape[1]} genes")
print(f"Samples: {adata.obs['sample_id'].nunique()}")
print(f"Conditions: {dict(adata.obs['condition'].value_counts())}")

  Loading dataset: /Users/shaunfchen/Documents/Repositories/spatial-microckg-agent/data/processed/tbi_preprocessed.h5ad
  Shape: 24922 spots × 19415 genes

Loaded: 24922 spots × 19415 genes
Samples: 6
Conditions: {0: np.int64(12746), 1: np.int64(12176)}


## 2.2 Run Stabl Feature Selection

In **condition mode** with **DE pre-filtering**, the pipeline uses a two-stage approach:

1. **Downsample** to ≤ 1,000 spots per sample (seed=42)
2. **ComBat batch correction** across the 6 samples
3. **t-test DE pre-filter** — Welch t-test between TBI and Sham, retain genes with BH-adjusted p < 0.01 and |log₂FC| > 0.5
4. **Ground-truth condition labels** read from `adata.obs['condition']`
5. **500-bootstrap Stabl** with L1-LogReg and FDP+ thresholding

This is **much faster** than the 2,000 HVG approach because the DE pre-filter typically selects only a few hundred to ~1,500 genes that are actually differentially expressed.

Results are cached by parameter hash — subsequent runs load instantly.

In [3]:
stabl_result = run_stabl_cached(
    adata,
    cache_dir=CACHE_DIR,
    dataset_name="geo_tbi",
    label_method="condition",
    n_bootstraps=500,
    prefilter="de",
    fdr_alpha=0.01,
    min_log2fc=0.5,
)

print(f"\nStabl results:")
print(f"  Features selected: {stabl_result['n_selected']}")
print(f"  FDP+ threshold: {stabl_result['threshold']:.4f}")
print(f"  Minimum FDP+: {stabl_result['fdr']:.4f}")

  No cache found — computing from scratch.
  Downsampled: 24922 → 6000 spots (≤1000 per sample)
  DE pre-filter (condition, t-test): 19415 → 735 genes (FDR < 0.01, |log2FC| > 0.5)
  Applying ComBat batch correction (6 batches)...


/Users/shaunfchen/Documents/Repositories/spatial-microckg-agent/.venv/lib/python3.11/site-packages/scanpy/preprocessing/_combat.py:344: RuntimeWarning: divide by zero encountered in divide
  (abs(g_new - g_old) / g_old).max(), (abs(d_new - d_old) / d_old).max()


  ComBat correction applied.
  Ground-truth condition labels: Sham(0)=3000, TBI(1)=3000


/Users/shaunfchen/Documents/Repositories/spatial-microckg-agent/.venv/lib/python3.11/site-packages/stabl/stabl.py:18: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm
/Users/shaunfchen/Documents/Repositories/spatial-microckg-agent/.venv/lib/python3.11/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


  Running Stabl (500 bootstraps, 735 features)...
  Stabl converged: 2 features selected
  FDP+ threshold: 0.9800, minimum FDP+: 1.0000
  Cached to /Users/shaunfchen/Documents/Repositories/spatial-microckg-agent/cache/stabl_results_afa34002d272.pkl and /Users/shaunfchen/Documents/Repositories/spatial-microckg-agent/cache/stabl_features_afa34002d272.csv

Stabl results:
  Features selected: 2
  FDP+ threshold: 0.9800
  Minimum FDP+: 1.0000


## 2.3 Review Selected Features

Stabl-certified biomarker genes ranked by stability score. A score of 1.0 means the gene was selected in every bootstrap iteration. These genes distinguish TBI from Sham conditions with controlled false-discovery guarantees.

In [4]:
import pandas as pd

df_features = pd.DataFrame({
    "Gene": stabl_result["selected_genes"],
    "Stability Score": [
        stabl_result["stability_scores"][g]
        for g in stabl_result["selected_genes"]
    ],
}).sort_values("Stability Score", ascending=False).reset_index(drop=True)

print(f"\nTop Stabl-certified biomarker genes ({len(df_features)} total):")
df_features.head(20)


Top Stabl-certified biomarker genes (2 total):


,Gene,Stability Score
0,Tmbim6,0.986
1,H3f3a,0.984


In [5]:
# Distribution of stability scores
print(f"Score statistics:")
print(f"  Mean: {df_features['Stability Score'].mean():.4f}")
print(f"  Median: {df_features['Stability Score'].median():.4f}")
print(f"  Min: {df_features['Stability Score'].min():.4f}")
print(f"  Max: {df_features['Stability Score'].max():.4f}")
print(f"  Genes with score = 1.0: {(df_features['Stability Score'] == 1.0).sum()}")

Score statistics:
  Mean: 0.9850
  Median: 0.9850
  Min: 0.9840
  Max: 0.9860
  Genes with score = 1.0: 0
